# ◈ Análise Gold — Feature Engineering & Modelos de IA
**Pipeline Metrópole SP · Arquitetura Medallion**

> A camada Gold contém **features prontas para ML** e os resultados dos modelos treinados.
> Este notebook analisa correlações, desempenho dos modelos e insights gerados.


In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': '#070b14', 'axes.facecolor': '#0d1526',
    'axes.edgecolor': '#1a3050',   'grid.color': '#1a3050',
    'text.color': '#d0e4ff',       'axes.labelcolor': '#7a9ab8',
    'xtick.color': '#4a6a8a',      'ytick.color': '#4a6a8a',
    'axes.titlecolor': '#00d4ff',  'axes.titlesize': 13,
    'axes.titleweight': 'bold',    'axes.grid': True,
    'figure.dpi': 120,
})
CYAN, GREEN, PURPLE = '#00d4ff', '#00ff88', '#7b2fff'
ORANGE, PINK, GOLD  = '#ff6b00', '#ff2d78', '#ffd700'
NEON = [CYAN, GREEN, PURPLE, ORANGE, PINK, GOLD]

# Detecta raiz do projeto
BASE = Path(os.environ.get('METRO_SP_BASE', Path.cwd()))
if not (BASE / 'data').exists():
    BASE = BASE.parent
BRONZE = BASE / 'data' / 'bronze'
SILVER = BASE / 'data' / 'silver'
GOLD   = BASE / 'data' / 'gold'
print(f"📂 Projeto: {BASE}")
print(f"   Bronze:  {BRONZE.exists()} | Silver: {SILVER.exists()} | Gold: {GOLD.exists()}")


In [ ]:
# ─── Carrega datasets Gold ────────────────────────────────────────────────────
g_anom  = pd.read_parquet(GOLD / 'air_quality_with_anomalies.parquet')
g_bus   = pd.read_parquet(GOLD / 'bus_demand_forecast.parquet')
g_occ   = pd.read_parquet(GOLD / 'occurrence_clusters.parquet')
g_clust = pd.read_parquet(GOLD / 'risk_clusters_summary.parquet') \
          if (GOLD / 'risk_clusters_summary.parquet').exists() else None

print("  Dataset                         Registros  Features")
print("  " + "-"*52)
for nome, df in [('air_quality + anomalias', g_anom),
                  ('bus demand forecast',     g_bus),
                  ('occurrence clusters',     g_occ)]:
    print(f"  {nome:<35} {len(df):>5,}  {len(df.columns):>7}")


In [ ]:
# ─── Correlação de Features — Qualidade do Ar ────────────────────────────────
feats_ar = ['mp10','mp25','o3','no2','co','so2',
            'temp_c','umidade_pct','vento_vel_ms',
            'z_mp10','z_mp25','z_o3','z_no2','iqar']
feats_ar = [c for c in feats_ar if c in g_anom.columns]

corr = g_anom[feats_ar].corr()

fig, ax = plt.subplots(figsize=(11, 9), facecolor='#070b14')
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

# Heatmap customizado (tema cyber)
import matplotlib.colors as mcolors
cmap = mcolors.LinearSegmentedColormap.from_list(
    'cyber', ['#ff2d78', '#070b14', '#00d4ff'])
im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label='Correlação de Pearson')

ax.set_xticks(range(len(corr))); ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr.columns, fontsize=9)
ax.set_title('Matriz de Correlação — Features Gold (Qualidade do Ar)',
             color=CYAN, fontweight='bold', pad=15)

for i in range(len(corr)):
    for j in range(len(corr)):
        val = corr.values[i, j]
        if abs(val) > 0.3:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=7, color='white' if abs(val) > 0.6 else '#d0e4ff')

plt.tight_layout()
plt.show()


In [ ]:
# ─── Isolation Forest — Análise de Anomalias ─────────────────────────────────
if 'anomalia_pred' in g_anom.columns:
    n_total = len(g_anom)
    n_anom  = g_anom['anomalia_pred'].sum()

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#070b14')
    fig.suptitle('Isolation Forest — Detecção de Anomalias (Qualidade do Ar)',
                 color=CYAN, fontweight='bold')

    # Score distribution
    axes[0].hist(g_anom[g_anom.anomalia_pred==0]['anomaly_score'], bins=40,
                 color=CYAN, alpha=0.7, label='Normal', density=True)
    axes[0].hist(g_anom[g_anom.anomalia_pred==1]['anomaly_score'], bins=20,
                 color=PINK, alpha=0.85, label='Anomalia', density=True)
    axes[0].set_title('Distribuição do Anomaly Score')
    axes[0].set_xlabel('Score (menor = mais anômalo)')
    axes[0].legend()

    # IQAr vs Score scatter
    axes[1].scatter(g_anom[g_anom.anomalia_pred==0]['iqar'],
                    g_anom[g_anom.anomalia_pred==0]['anomaly_score'],
                    color=CYAN, alpha=0.3, s=8, label='Normal')
    axes[1].scatter(g_anom[g_anom.anomalia_pred==1]['iqar'],
                    g_anom[g_anom.anomalia_pred==1]['anomaly_score'],
                    color=PINK, alpha=0.8, s=20, label='Anomalia', zorder=5)
    axes[1].set_title('IQAr vs Anomaly Score')
    axes[1].set_xlabel('IQAr'); axes[1].set_ylabel('Anomaly Score')
    axes[1].legend()

    # Por estação
    if 'estacao_id' in g_anom.columns:
        est_anom = g_anom.groupby('estacao_id')['anomalia_pred'].mean() * 100
        bars = axes[2].bar(est_anom.index, est_anom.values,
                           color=[PINK if v > 5 else CYAN for v in est_anom.values],
                           alpha=0.85)
        axes[2].axhline(5, color=ORANGE, linestyle='--', linewidth=1.2,
                        label='Threshold 5%')
        axes[2].set_title('Taxa de Anomalia por Estação (%)')
        axes[2].legend()
        axes[2].tick_params(axis='x', rotation=30, labelsize=8)

    plt.tight_layout()
    plt.show()
    print(f"\n  Anomalias detectadas: {n_anom} / {n_total} ({n_anom/n_total:.1%})")
    print(f"  Contamination param: 5%  |  n_estimators: 150")


In [ ]:
# ─── XGBoost — Análise de Demanda de Transporte ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#070b14')
fig.suptitle('XGBoost — Previsão de Demanda de Transporte', color=CYAN, fontweight='bold')

# Distribuição do target
axes[0].hist(g_bus['target'].dropna(), bins=30, color=PURPLE, alpha=0.85, edgecolor='none')
axes[0].axvline(g_bus['target'].mean(), color=CYAN, linestyle='--', linewidth=2,
                label=f"Média={g_bus['target'].mean():.2f}")
axes[0].set_title('Distribuição do Target (lotação t+1h)')
axes[0].legend(fontsize=8)

# Target por hora do dia
agg_hora = g_bus.groupby('hora_do_dia')['target'].agg(['mean','std']).reset_index()
axes[1].fill_between(agg_hora['hora_do_dia'],
                     agg_hora['mean'] - agg_hora['std'],
                     agg_hora['mean'] + agg_hora['std'],
                     alpha=0.2, color=CYAN)
axes[1].plot(agg_hora['hora_do_dia'], agg_hora['mean'],
             color=CYAN, linewidth=2.5, marker='o', markersize=5)
axes[1].set_title('Lotação Média por Hora do Dia')
axes[1].set_xlabel('Hora'); axes[1].set_ylabel('Lotação (0–1)')
axes[1].set_xticks(range(0, 24, 2))

# Correlação features vs target
feat_cols = ['hora_do_dia','dia_semana','e_feriado','e_fim_semana',
             'lag_1h','lag_24h','temperatura_c','precipitacao_mm']
feat_cols = [c for c in feat_cols if c in g_bus.columns]
corrs = g_bus[feat_cols + ['target']].corr()['target'].drop('target').abs().sort_values(ascending=True)
axes[2].barh(corrs.index, corrs.values,
             color=[CYAN if v > 0.5 else PURPLE for v in corrs.values], alpha=0.85)
axes[2].set_title('Correlação Features × Target')
axes[2].set_xlabel('|Pearson|')

plt.tight_layout()
plt.show()
print(f"\n  Feature mais correlacionada: {corrs.idxmax()} ({corrs.max():.3f})")


In [ ]:
# ─── DBSCAN — Análise dos Clusters de Risco ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='#070b14')
fig.suptitle('DBSCAN — Mapa de Risco Urbano', color=CYAN, fontweight='bold')

CLR = {'INFRAESTRUTURA': CYAN, 'MEIO_AMBIENTE': GREEN,
       'MOBILIDADE': ORANGE, 'SEGURANCA': PINK, 'OUTROS': '#607090'}

df_geo = g_occ.dropna(subset=['lat','lon'])
noise  = df_geo[df_geo.cluster_id == -1]
clust  = df_geo[df_geo.cluster_id >= 0]

# Mapa geo
for cat, grp in clust.groupby('categoria_l1'):
    axes[0].scatter(grp['lon'], grp['lat'],
                    c=CLR.get(cat,'#888'), s=12, alpha=0.7, label=cat)
axes[0].scatter(noise['lon'], noise['lat'],
                c='#2a4060', s=5, alpha=0.4, label='Ruído')
if g_clust is not None and 'lon_centro' in g_clust.columns:
    axes[0].scatter(g_clust['lon_centro'], g_clust['lat_centro'],
                    marker='*', s=200, c=GOLD, zorder=10, label='Centroide')
axes[0].set_title('Clusters de Risco (eps=0.008, min=5)')
axes[0].legend(fontsize=7, loc='lower right')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Tamanho dos clusters
if g_clust is not None and 'n_ocorrencias' in g_clust.columns:
    g_clust_sorted = g_clust.sort_values('n_ocorrencias', ascending=True)
    dom = g_clust_sorted.get('categoria_dom', pd.Series(['OUTROS']*len(g_clust_sorted)))
    colors_cl = [CLR.get(c, '#888') for c in dom]
    axes[1].barh(range(len(g_clust_sorted)), g_clust_sorted['n_ocorrencias'],
                 color=colors_cl, alpha=0.85, edgecolor='none')
    axes[1].set_yticks(range(len(g_clust_sorted)))
    axes[1].set_yticklabels(g_clust_sorted.get('cluster_id', range(len(g_clust_sorted))),
                            fontsize=9)
    axes[1].set_title('Ocorrências por Cluster')
    axes[1].set_xlabel('Nº de ocorrências')

plt.tight_layout()
plt.show()

n_cl = (g_occ.cluster_id >= 0).sum()
n_no = (g_occ.cluster_id == -1).sum()
print(f"\n  Pontos clusterizados: {n_cl} ({n_cl/len(g_occ):.1%})")
print(f"  Ruído (outliers):     {n_no} ({n_no/len(g_occ):.1%})")
if g_clust is not None:
    print(f"  Clusters detectados: {len(g_clust)}")


In [ ]:
# ─── Sumário Executivo — Camada Gold ─────────────────────────────────────────
print("\n" + "=" * 65)
print("  SUMÁRIO EXECUTIVO — PIPELINE METRÓPOLE SP")
print("=" * 65)

print("\n  CAMADA GOLD — DATASETS GERADOS:")
for nome, df in [('air_quality_anomaly',    g_anom),
                  ('bus_demand_forecast',    g_bus),
                  ('occurrence_clusters',    g_occ)]:
    print(f"    📁 {nome:<30}  {len(df):>5,} registros  ×  {len(df.columns):>3} features")

print("\n  MODELOS TREINADOS:")
if 'anomalia_pred' in g_anom.columns:
    n_a = g_anom['anomalia_pred'].sum()
    print(f"    ⚡ Isolation Forest   → {n_a} anomalias ({n_a/len(g_anom):.1%})")
print(f"    📈 XGBoost            → {len(g_bus)} amostras, split 80/20 temporal")
if g_clust is not None:
    print(f"    📍 DBSCAN             → {len(g_clust)} clusters de risco")

print("\n  QUALIDADE DO AR:")
if 'faixa_iqar' in g_anom.columns:
    for faixa, cnt in g_anom['faixa_iqar'].value_counts().items():
        pct = cnt/len(g_anom)*100
        bar = "█" * int(pct // 5)
        print(f"    {faixa:<15} [{bar:<20}] {cnt:>4} ({pct:.1f}%)")

print("\n  → Ver dashboard: http://localhost:8501")
print("=" * 65)
